# Notebook 03 — Geographic Diversity

### Obiettivi

In questo notebook viene costruita la metrica di **diversità geografica dei consumi cinematografici**, la terza dimensione prevista dal progetto di analisi del fenomeno del diversity decay.

L'obiettivo è estendere l'analisi della diversità degli utenti oltre la dimensione dei generi cinematografici, valutando se nel corso del tempo gli utenti mantengano una distribuzione ampia dei contenuti anche rispetto ai **paesi di produzione dei film**.

La metrica geografica viene costruita seguendo un approccio analogo a quello utilizzato nel Notebook 01 per la diversità dei generi:

- recuperare il paese di produzione di ciascun film attraverso il mapping IMDb–TMDb
- associare ogni film consumato al relativo paese di produzione
- costruire, per ogni coppia utente–anno, la distribuzione dei consumi per paese
- calcolare un indice di diversità geografica basato sull'entropia di Shannon

Il notebook ha quindi i seguenti obiettivi principali:
- costruire il mapping `movieId → country` utilizzando le informazioni disponibili tramite IMDb e TMDb
- verificare la copertura geografica del catalogo e dei consumi effettivi degli utenti
- creare il dataset longitudinale `user_year_geo_longitudinal.parquet`
- ottenere una nuova metrica (`geo_entropy`) comparabile con l'entropia dei generi già utilizzata nelle analisi precedenti

La metrica costruita verrà successivamente integrata nelle analisi statistiche finali insieme alle altre dimensioni della diversity:
- diversità dei generi (`entropy`)
- esplorazione di nuovi contenuti (`novelty`)
- diversità geografica (`geo_entropy`)

Questo permetterà di verificare se eventuali fenomeni di restringimento delle preferenze degli utenti siano presenti non solo nella dimensione tematica dei generi, ma anche nella provenienza geografica dei contenuti consumati.

#### Import delle librerie

In [1]:
import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path
from tqdm import tqdm

print("Ambiente configurato correttamente")

Ambiente configurato correttamente


#### Definizione dei path

In [2]:
project_root = Path.cwd().parent

data_path = project_root / "data"
processed_path = data_path / "processed"

print("Project root:", project_root)
print("Processed path:", processed_path)

Project root: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens
Processed path: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/data/processed


#### Chiave API
Per recuperare le informazioni geografiche dei film viene utilizzata la TMDb API.
La chiave API viene utilizzata esclusivamente per interrogare i metadati dei film tramite identificativo IMDb.

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

#verifica lettura
if not TMDB_API_KEY:
    print("TMDB_API_KEY non trovata! Controlla il file .env")
else:
    print("Chiave caricata con successo!")

Chiave caricata con successo!


### Caricamento dei dataset

Vengono caricati:

- il dataset dei rating MovieLens (`ratings.csv`);
- il mapping creato nel notebook di setup (`imdb_mapping.parquet`).

Il file `imdb_mapping.parquet` contiene il collegamento tra gli identificativi
MovieLens e gli identificativi IMDb dei film.

Questo mapping verrà utilizzato per recuperare tramite TMDb i metadati relativi
ai paesi di produzione.

#### Caricamento ratings

In [4]:
ratings_path = data_path / "raw" / "ml-25m" / "ratings.csv"
ratings = pd.read_csv(
    ratings_path
)

ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [5]:
ratings.shape

(25000095, 4)

#### Caricamento mapping IMDb

In [6]:
imdb_mapping_path = processed_path / "imdb_mapping.parquet"

imdb_mapping = pd.read_parquet(
    imdb_mapping_path
)

imdb_mapping.head()

,movieId,imdbId_str
0,1,tt0114709
1,2,tt0113497
2,3,tt0113228
3,4,tt0114885
4,5,tt0113041


In [7]:
imdb_mapping.shape

(62423, 2)

In [8]:
imdb_mapping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62423 entries, 0 to 62422
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   movieId     62423 non-null  int64 
 1   imdbId_str  62423 non-null  object
dtypes: int64(1), object(1)
memory usage: 975.5+ KB


### Controllo copertura mapping
Prima di interrogare TMDb viene verificata la copertura del mapping IMDb rispetto
al catalogo MovieLens.

Questo controllo permette di identificare eventuali film privi di identificativo
IMDb e quindi non recuperabili tramite API.

In [9]:
coverage = (
    ratings["movieId"]
    .isin(imdb_mapping["movieId"])
    .mean()
)

print(f"Copertura mapping IMDb: {coverage:.2%}")

Copertura mapping IMDb: 100.00%


#### Controllo duplicati

In [10]:
imdb_mapping["movieId"].duplicated().sum()

np.int64(0)

### Recupero dei paesi di produzione tramite TMDb API

Il file `imdb_mapping.parquet` contiene il collegamento tra gli identificativi
MovieLens (`movieId`) e gli identificativi IMDb dei film.

Per ottenere le informazioni geografiche viene utilizzata la TMDb API.

Il processo è composto da due passaggi:

1. ricerca del film tramite identificativo IMDb;
2. recupero dei metadati del film, inclusi i paesi di produzione.

Per ogni film viene estratto il campo `production_countries`.

Poiché alcuni film possono essere prodotti da più paesi, nella prima implementazione
viene mantenuto il primo paese disponibile restituito dalla fonte.

#### Funzione per interrogare TMDb

In [11]:
def get_tmdb_movie_id_from_imdb(imdb_id, api_key):
    # Recupera il TMDb id a partire dall'IMDb id

    url = f"https://api.themoviedb.org/3/find/{imdb_id}"

    params = {
        "api_key": api_key,
        "external_source": "imdb_id"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        if response.status_code != 200:
            return None

        data = response.json()

        if len(data.get("movie_results", [])) == 0:
            return None

        return data["movie_results"][0]["id"]

    except Exception:
        return None

#### Funzione per recuperare il paese

In [12]:
def get_production_country(tmdb_id, api_key):
    
    # Recupera il paese di produzione da TMDb

    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

    params = {
        "api_key": api_key
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        if response.status_code != 200:
            return None


        data = response.json()


        countries = data.get(
            "production_countries",
            []
        )


        if len(countries) == 0:
            return None


        return countries[0]["name"]


    except Exception:
        return None

In [13]:
# test per controllare il funzionamento delle funzioni di recupero dei dati da TMDb
test_imdb = imdb_mapping.iloc[0]["imdbId_str"]

test_imdb

'tt0114709'

In [14]:
tmdb_test = get_tmdb_movie_id_from_imdb(
    test_imdb,
    TMDB_API_KEY
)

tmdb_test

862

In [15]:
country_test = get_production_country(
    tmdb_test,
    TMDB_API_KEY
)

country_test

'United States of America'

### Costruzione del mapping movieId → paese
Viene costruito un nuovo dataset contenente il collegamento:

movieId → paese di produzione

Questo file verrà salvato localmente per evitare di ripetere le chiamate
alla TMDb API nei notebook successivi.

In [16]:
movies_country = imdb_mapping.copy()

movies_country.head()

,movieId,imdbId_str
0,1,tt0114709
1,2,tt0113497
2,3,tt0113228
3,4,tt0114885
4,5,tt0113041


In [17]:
movies_country["country"] = None
movies_country.head()

,movieId,imdbId_str,country
0,1,tt0114709,None
1,2,tt0113497,None
2,3,tt0113228,None
3,4,tt0114885,None
4,5,tt0113041,None


#### Recupero

In [18]:
results = []

for idx, row in tqdm(
    movies_country.iterrows(),
    total=len(movies_country)
):

    imdb_id = row["imdbId_str"]

    tmdb_id = get_tmdb_movie_id_from_imdb(
        imdb_id,
        TMDB_API_KEY
    )


    if tmdb_id is not None:

        country = get_production_country(
            tmdb_id,
            TMDB_API_KEY
        )

    else:
        country = None


    results.append(country)


    # pausa per evitare troppe richieste
    time.sleep(0.05)

100%|██████████| 62423/62423 [12:22:05<00:00,  1.40it/s]  


#### Aggiungiamo i risultati al database

In [19]:
movies_country["country"] = results

In [20]:
movies_country.head()

,movieId,imdbId_str,country
0,1,tt0114709,United States of America
1,2,tt0113497,United States of America
2,3,tt0113228,United States of America
3,4,tt0114885,United States of America
4,5,tt0113041,United States of America


### Analisi della copertura geografica

Prima di procedere alla costruzione della metrica viene verificata la percentuale
di film per cui è stato possibile recuperare il paese di produzione.

I film senza informazione geografica verranno esclusi dal calcolo della metrica.

In [21]:
country_coverage = (
    movies_country["country"]
    .notna()
    .mean()
)

print(
    f"Copertura geografica: {country_coverage:.2%}"
)

Copertura geografica: 95.15%


**Copertura del mapping geografico**

Il recupero del paese di produzione tramite TMDb ha consentito di associare un paese al 95.15% dei film presenti nel mapping IMDb.
Successivamente il mapping viene unito al dataset completo dei rating.
Poiché i film senza paese rappresentano una quota molto ridotta delle osservazioni, essi vengono esclusi dalle analisi successive.

#### Controllo dei valori mancanti

In [22]:
movies_country["country"].isna().sum()

np.int64(3026)

#### Merge dei rating con il paese di produzione

In [28]:
ratings_geo = ratings.merge(
    movies_country[["movieId", "country"]],
    on="movieId",
    how="left"
)

ratings_geo["year"] = pd.to_datetime(ratings_geo["timestamp"], unit="s").dt.year

coverage_ratings = ratings_geo["country"].notna().mean() * 100

print("Osservazioni:", len(ratings_geo))
print("Country mancanti:", ratings_geo["country"].isna().sum())
print(
    "Copertura sui rating:",
    round(
        100*(1-ratings_geo["country"].isna().mean()),
        2
    ),
    "%"
)

ratings_geo.head()

Osservazioni: 25000095
Country mancanti: 69283
Copertura sui rating: 99.72 %


,userId,movieId,rating,timestamp,country,year
0,1,296,5.0,1147880044,United States of America,2006
1,1,306,3.5,1147868817,France,2006
2,1,307,5.0,1147868828,France,2006
3,1,665,5.0,1147878820,Yugoslavia,2006
4,1,899,3.5,1147868510,United States of America,2006


### Salvataggio mapping
Il mapping geografico ottenuto viene salvato in formato Parquet.
Questo file rappresenta il collegamento definitivo tra il catalogo MovieLens e la provenienza geografica dei film.

In [29]:
country_mapping_path = (processed_path / "movie_country_mapping.parquet")

movies_country.to_parquet(
    country_mapping_path,
    index=False
)

print(
    "Salvato:",
    country_mapping_path
)

Salvato: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/data/processed/movie_country_mapping.parquet


### Costruzione metrica geografica

#### Eliminiamo i film senza paese

In [30]:
ratings_geo = ratings_geo.dropna(subset=["country"]).copy()

#### Conteggio dei film per utente-anno-paese

In [31]:
user_year_country = (
    ratings_geo
    .groupby(["userId", "year", "country"])
    .size()
    .reset_index(name="n_films")
)

user_year_country.head()

,userId,year,country,n_films
0,1,2006,Argentina,2
1,1,2006,Australia,1
2,1,2006,Brazil,1
3,1,2006,Canada,1
4,1,2006,Denmark,2


#### Calcolo delle quote

In [32]:
user_year_country["share"] = (
    user_year_country["n_films"] /
    user_year_country.groupby(
        ["userId","year"]
    )["n_films"].transform("sum")
)

#### Entropia geografica di Shannon

**Costruzione della metrica di diversità geografica**

Per ciascuna coppia utente–anno viene costruita la distribuzione dei film consumati rispetto al paese di produzione.
Su tale distribuzione viene calcolata l'Entropia di Shannon, ottenendo una misura della varietà geografica dei contenuti consumati.
Valori elevati indicano utenti che guardano film provenienti da numerosi paesi con una distribuzione relativamente equilibrata, mentre valori bassi indicano consumi concentrati su pochi paesi.

In [33]:
from scipy.stats import entropy

geo_entropy = (
    user_year_country
    .groupby(["userId", "year"])["share"]
    .apply(lambda x: entropy(x, base=2))
    .reset_index(name="geo_entropy")
)

geo_entropy.head()

,userId,year,geo_entropy
0,1,2006,3.418980
1,2,2006,1.368653
2,3,2015,1.902675
3,3,2016,0.969918
4,3,2017,2.315034


#### Numero dei paesi consumati

In [34]:
geo_counts = (
    user_year_country
    .groupby(["userId","year"])
    .size()
    .reset_index(name="n_countries")
)

geo_counts.head()

,userId,year,n_countries
0,1,2006,18
1,2,2006,12
2,3,2015,23
3,3,2016,4
4,3,2017,12


#### Datset finale

In [35]:
user_year_geo = geo_entropy.merge(
    geo_counts,
    on=["userId","year"],
    how="left"
)

user_year_geo.head()

,userId,year,geo_entropy,n_countries
0,1,2006,3.418980,18
1,2,2006,1.368653,12
2,3,2015,1.902675,23
3,3,2016,0.969918,4
4,3,2017,2.315034,12


#### Controllo NaN finale

In [36]:
user_year_geo.isna().sum()

userId         0
year           0
geo_entropy    0
n_countries    0
dtype: int64

In [37]:
user_year_geo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240347 entries, 0 to 240346
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   userId       240347 non-null  int64  
 1   year         240347 non-null  int32  
 2   geo_entropy  240347 non-null  float64
 3   n_countries  240347 non-null  int64  
dtypes: float64(1), int32(1), int64(2)
memory usage: 6.4 MB


In [38]:
user_year_geo.shape

(240347, 4)

#### Salvataggio dataset finale

In [39]:
output_path = (
    processed_path /
    "user_year_geo_longitudinal.parquet"
)

user_year_geo.to_parquet(
    output_path,
    index=False
)

print("Salvato in:")
print(output_path)

Salvato in:
/Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/data/processed/user_year_geo_longitudinal.parquet
